In [ ]:
import csv
import math
import threading
import time
from dataclasses import dataclass
from queue import Queue, Empty

import numpy as np

# Lets connect to Beamng and extract levels and scenarios infos
from beamngpy import BeamNGpy, Scenario, Vehicle
from beamngpy.api.beamng import GEVehiclesApi

In [ ]:
from beamngpy import BeamNGpy, Scenario, Vehicle, set_up_simple_logging
from beamngpy.sensors import RoadsSensor

In [ ]:
import matplotlib
%matplotlib qt
# matplotlib.use("qtAgg")  # Or another interactive backend like "TkAgg", Qt5Agg
# matplotlib.use("widget")  # Or another interactive backend like "TkAgg", Qt5Agg
import matplotlib.pyplot as plt

In [ ]:
# beamng:
#   host: 127.0.0.1
#   port: 25252
#   open_args:
#     launch: False
#     debug: False
#     extensions:
#       - xlab/xlabCore

beamng_info = {
    'host': '127.0.0.1',
    'port': 25252
}
launch_info = {
    'launch': False,
    'debug': False,
    'extensions': ['xlab/xlabCore',]
}
beamng = BeamNGpy(**beamng_info)
beamng.open(**launch_info)

In [ ]:
# # List all levels in BeamNG
# levels, scenarios = beamng.scenario.get_levels_and_scenarios()
# for k, v in levels.items():
#     print(f"Level: {k}, Info: {v}")

In [ ]:
# # Print the scenarios for each level
# for level, scen_list in scenarios.items():
#     print(f"Scenarios in Level '{level}':")
#     for scen in scen_list:
#         print(f"{scen}")

In [ ]:
# current_level = 'small_island'
# # current_level = 'smalldriftisland'
# # Load the scenario
# # beamng.scenario.load_scenario(current_level, 'west_coast_usa', reset=True)
# data = dict(type="StartLevel")
# data["levelName"] = current_level
# beamng._send(data).ack("StartedLevel")

In [ ]:
# Let's get current vehicle
curr_vehicles = beamng.get_current_vehicles(include_config=False)
curr_vehicles

In [ ]:
polling_rate = 0.5  # Hz
veh = curr_vehicles[list(curr_vehicles.keys())[0]]
veh.connect(beamng)
rs = RoadsSensor("rs1", beamng, veh, physics_update_time=polling_rate)

In [ ]:
# ===========================
# ---- Small utilities  -----
# ===========================

def angle_wrap(a):
    return a # (a + math.pi) % (2*math.pi) - math.pi

def rotation(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s],[s, c]], dtype=float)

def kappa_from_three_points(P0, P1, P2, eps=1e-12):
    (x0,y0),(x1,y1),(x2,y2) = P0, P1, P2
    abx, aby = (x1-x0), (y1-y0)
    acx, acy = (x2-x0), (y2-y0)
    bcx, bcy = (x2-x1), (y2-y1)
    ab = math.hypot(abx, aby) or eps
    bc = math.hypot(bcx, bcy) or eps
    ac = math.hypot(acx, acy) or eps
    cross = abx*acy - aby*acx
    return cross / (ab * bc * ac)


# ===================================
# ---- Param cubic helpers  ---------
# ===================================

def eval_param_poly3(aU,bU,cU,dU, aV,bV,cV,dV, p):
    u  = aU + bU*p + cU*p**2 + dU*p**3
    v  = aV + bV*p + cV*p**2 + dV*p**3
    up = bU + 2*cU*p + 3*dU*p**2
    vp = bV + 2*cV*p + 3*dV*p**2
    upp= 2*cU + 6*dU*p
    vpp= 2*cV + 6*dV*p
    return (u,v), (up,vp), (upp,vpp)

def curvature_from_uv_derivs(up,vp, upp,vpp, eps=1e-12):
    num = up*vpp - vp*upp
    den = (up*up + vp*vp)**1.5 + eps
    return num / den

def uv_to_world(xStart, yStart, u, v, theta0=0.0):
    if theta0 == 0.0:
        return float(xStart + u), float(yStart + v)

    R = rotation(theta0)
    xw, yw = (np.array([xStart, yStart]) + R @ np.array([u, v])).tolist()
    return float(xw), float(yw)


# ===========================
# ---- Data structure   -----
# ===========================

@dataclass
class TrackSnapshot:
    # Centerline world pose (closest-ish)
    x_cl: float
    y_cl: float
    z_cl: float
    heading_path: float
    
    # Curvatures
    kappa: float
    radius: float
    
    # Widths (centerline -> edges)
    w_left: float
    w_right: float
    halfWidth: float
    
    x_left: float
    y_left: float
    x_right: float
    y_right: float
    
    # s_along: float               # cumulative arclength (approx)
    # p_star: float                # param on [0,1] of the segment
    seg_id: int                  # our rolling segment id
        
        
# =====================================================
# ---- Operation on the cubic projection  -------------
# =====================================================

def discrete_closest_on_poly(
    coeffs_u, coeffs_v, xStart, yStart, p_grid,
    theta0=0, ret_all=False,
):
    xs, ys = [], []
    kappas = []
    headings = []
    for p in p_grid:
        (u,v),(up,vp), (upp, vpp) = eval_param_poly3(*coeffs_u, *coeffs_v, p)
        # Extract world point
        xw, yw = uv_to_world(xStart, yStart, u, v, theta0)
        xs.append(xw); ys.append(yw)
        
        if ret_all:
            # Extract curvature
            kappa = curvature_from_uv_derivs(up, vp, upp, vpp)
            kappas.append(kappa)
            # Extract heading
            heading = angle_wrap(theta0 + math.atan2(vp, up))
            headings.append(heading)
    
    xs = np.array(xs)
    ys = np.array(ys)
    return (xs, ys, kappas, headings) if ret_all else (xs, ys)


# =====================================================
# ---- One-tick snapshot computation  -----------------
# =====================================================

def compute_snapshot(
    rd, existing_seg_ids, grid_N=101
):
    # Coeffs & segment anchors
    coeffs_u_cl = (rd["uAofCL"], rd["uBofCL"], rd["uCofCL"], rd["uDofCL"])
    coeffs_v_cl = (rd["vAofCL"], rd["vBofCL"], rd["vCofCL"], rd["vDofCL"])

    # xPos0, yPos0, zPos0 = float(rd["xP0onCL"]), float(rd["yP0onCL"]), float(rd["zP0onCL"])
    xStart = rd["xStartCL"]; yStart = rd["yStartCL"]
    xStartL = rd["xStartL"]; yStartL = rd["yStartL"]
    xStartR = rd["xStartR"]; yStartR = rd["yStartR"]
    

    # Compute via hash unique segment id based on start point
    seg_id = hash( (round(xStart,3), round(yStart,3)) )
    if seg_id not in existing_seg_ids:
        print(f"[tracker] New segment at start ({xStart:.1f}, {yStart:.1f}), id={seg_id}")
    else:
        return []  # already processed
    existing_seg_ids.add(seg_id)
    
    # p grid
    p_grid = np.linspace(0.0, 1.0, grid_N)
    
    # Center line points for this segment
    xCLs, yCLs, kappas, headings = discrete_closest_on_poly(
        coeffs_u_cl, coeffs_v_cl, xStart, yStart, p_grid,
        theta0=0, ret_all=True
    )
    
    # # Find the index closest to our current position
    # dists = np.hypot(xCLs - xPos0, yCLs - yPos0)
    # p_star = int(np.argmin(dists))
    # xCLs = xCLs[p_star:p_star+1]
    # yCLs = yCLs[p_star:p_star+1]
    # kappas = kappas[p_star:p_star+1]
    # headings = headings[p_star:p_star+1]
    
    # Left edge points for this segment
    coeffs_u_L = (rd["uAofLeftRE"], rd["uBofLeftRE"], rd["uCofLeftRE"], rd["uDofLeftRE"])
    coeffs_v_L = (rd["vAofLeftRE"], rd["vBofLeftRE"], rd["vCofLeftRE"], rd["vDofLeftRE"])
    xLs, yLs = discrete_closest_on_poly(
        coeffs_u_L, coeffs_v_L, xStartL, yStartL, p_grid,
        theta0=0, ret_all=False
    )
    # xLs = xLs[p_star:p_star+1]
    # yLs = yLs[p_star:p_star+1]
    
    # Right edge points for this segment
    coeffs_u_R = (rd["uAofRightRE"], rd["uBofRightRE"], rd["uCofRightRE"], rd["uDofRightRE"])
    coeffs_v_R = (rd["vAofRightRE"], rd["vBofRightRE"], rd["vCofRightRE"], rd["vDofRightRE"])
    xRs, yRs = discrete_closest_on_poly(
        coeffs_u_R, coeffs_v_R, xStartR, yStartR, p_grid,
        theta0=0, ret_all=False
    )
    # xRs = xRs[p_star:p_star+1]
    # yRs = yRs[p_star:p_star+1]
    
    
    # Let's build the sequence of TrackSnapshot for this segment
    snapshots = []
    for i in range(len(xCLs)):
        x_cl, y_cl = xCLs[i], yCLs[i]
        heading_cl = headings[i]
        kappa = kappas[i]
        radius = (1.0/abs(kappa)) if abs(kappa)>1e-9 else math.inf
        
        x_left, y_left = xLs[i], yLs[i]
        x_right, y_right = xRs[i], yRs[i]
        
        w_left = math.hypot(x_left - x_cl, y_left - y_cl)
        w_right= math.hypot(x_right - x_cl, y_right - y_cl)
        halfWidth = 0.5 * (w_left + w_right)
        
        # s_along = float(rd.get("s_along", float("nan"))) + p_grid[i] * float(rd.get("segLength", float("nan")))
        # p_star = p_grid[i]
        
        snap = TrackSnapshot(
            # t=float(rd.get("time", time.time())),
            x_cl=x_cl, y_cl=y_cl, z_cl=float(rd.get("zP0onCL", 0.0)),
            heading_path=heading_cl,
            kappa=kappa,
            radius=radius,
            w_left=w_left, w_right=w_right,
            halfWidth=halfWidth,
            x_left=x_left,
            y_left=y_left,
            x_right=x_right,
            y_right=y_right,
            seg_id=seg_id,
            # s_along=s_along,
            # p_star=p_star
        )
        snapshots.append(snap)

    return snapshots

In [ ]:

# ====================================
# ---- Polling + Live Plot (Qt)  -----
# ====================================

class SensorPoller(threading.Thread):
    def __init__(self, sensor, poll_period_sec, out_queue, stop_event):
        super().__init__(daemon=True)
        self.sensor = sensor
        self.dt = float(poll_period_sec)
        self.q = out_queue
        self.stop_event = stop_event

    def run(self):
        next_t = time.perf_counter()
        while not self.stop_event.is_set():
            try:
                data = self.sensor.poll()
                # print(f"[poller] got data: {data }")
                if data is not None and len(data) > 0:
                    self.q.put(data[0.0])  # data is a list of dicts
            except Exception as e:
                print(f"[poller] error: {e!r}")
            next_t += self.dt
            sleep = next_t - time.perf_counter()
            if sleep > 0:
                time.sleep(sleep)
            else:
                # we're behind; skip sleep to catch up
                next_t = time.perf_counter()

def run_logger_and_plot(
    sensor, road_poll_sec=0.05, grid_N=101
):
    """
    Segment-aware polling + realtime plot.
    Designed for Jupyter with `%matplotlib qt` (interactive Qt backend).
    No CSV in this variant (kept commented below).
    """
    q = Queue(maxsize=512)
    stop_event = threading.Event()
    poller = SensorPoller(sensor, road_poll_sec, q, stop_event)
    poller.start()

    # # CSV (optional — uncomment if you want logging here)
    # fieldnames = [
    #     "time","seg_id",
    #     "x_cl","y_cl","z_cl",
    #     "heading_cl_discrete","heading_cl_poly",
    #     "kappa_discrete","kappa_poly","radius_discrete","radius_poly",
    #     "w_left","w_right","halfWidth",
    #     "dist2CL","dist2Left","dist2Right",
    #     "speedLimit","drivability","flag1way","numlane",
    #     "s_along","p_star_discrete","p_star_poly"
    # ]
    # f = open("track_log.csv", "w", newline="")
    # writer = csv.DictWriter(f, fieldnames=fieldnames)
    # writer.writeheader()

    # Figure/window
    fig, ax = plt.subplots(figsize=(7, 7))
    line_cl= ax.scatter([], [], s=2, c='g', label="centerline")
    line_left= ax.scatter([], [], s=1, c='r', label="left edge")
    line_right = ax.scatter([], [], s=1, c='b', label="right edge")
    ax.set_aspect("equal")
    ax.set_title("Live Centerline (segment-aware)")
    ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
    ax.legend(loc="best")

    # ensure the GUI event loop runs but doesn't block the cell
    plt.show(block=False)

    # closing the window stops everything
    def _on_close(_evt):
        stop_event.set()
    fig.canvas.mpl_connect('close_event', _on_close)

    xs_cl, ys_cl = [], []
    xs_left, ys_left = [], []
    xs_right, ys_right = [], []

    # last_s = 0.0
    last_plot_anchor = None
    existing_seg_ids = set()
    last_segment_start = None
    last_segment_end = None
    last_segment_size = 0

    try:
        while not stop_event.is_set():
            # pull any available sensor packet without hanging UI
            try:
                rd = q.get(timeout=0.05)
            except Empty:
                # keep GUI responsive even when no data
                fig.canvas.flush_events()
                continue

            snaps = compute_snapshot(
                rd, existing_seg_ids, grid_N=grid_N
            )
            
            if len(snaps) == 0:
                continue

            # # CSV (optional)
            # writer.writerow({
            #     "time": snap.t, "seg_id": snap.seg_id,
            #     "x_cl": snap.x_cl, "y_cl": snap.y_cl, "z_cl": snap.z_cl,
            #     "heading_cl_discrete": snap.heading_cl_discrete,
            #     "heading_cl_poly": snap.heading_cl_poly,
            #     "kappa_discrete": snap.kappa_discrete, "kappa_poly": snap.kappa_poly,
            #     "radius_discrete": snap.radius_discrete, "radius_poly": snap.radius_poly,
            #     "w_left": snap.w_left, "w_right": snap.w_right, "halfWidth": snap.halfWidth,
            #     "dist2CL": snap.dist2CL, "dist2Left": snap.dist2Left, "dist2Right": snap.dist2Right,
            #     "speedLimit": snap.speedLimit, "drivability": snap.drivability,
            #     "flag1way": snap.flag1way, "numlane": snap.numlane,
            #     "s_along": snap.s_along,
            #     "p_star_discrete": snap.p_star_discrete, "p_star_poly": snap.p_star_poly
            # })
            # if (len(xs_cl) % 50) == 0:
            #     f.flush()
            
            # current_segment_start = (snaps[0].x_cl, snaps[0].y_cl)
            # current_segment_end = (snaps[-1].x_cl, snaps[-1].y_cl)
            
            # # Figure how to insert the new segment points?
            # append_last = True
            # if last_segment_start is None or last_segment_end is None:
            #     append_last = True
            # else:
            #     # Check if we should append at the end or insert at the beginning
            #     dist_start_to_last_end = math.hypot(
            #         current_segment_start[0] - last_segment_end[0],
            #         current_segment_start[1] - last_segment_end[1]
            #     )
            #     dist_end_to_last_start = math.hypot(
            #         current_segment_end[0] - last_segment_start[0],
            #         current_segment_end[1] - last_segment_start[1]
            #     )
            #     if dist_start_to_last_end < dist_end_to_last_start:
            #         append_last = True
            #     else:
            #         append_last = False
            # last_segment_start = (snaps[0].x_cl, snaps[0].y_cl)
            # last_segment_end = (snaps[-1].x_cl, snaps[-1].y_cl)
            
            # if not append_last:
            #     # Let's reverse the snaps to keep ordering consistent
            #     snaps.reverse()
                
            xs_cl.extend([snap.x_cl for snap in snaps])
            ys_cl.extend([snap.y_cl for snap in snaps])
            xs_left.extend([snap.x_left for snap in snaps])
            ys_left.extend([snap.y_left for snap in snaps])
            xs_right.extend([snap.x_right for snap in snaps])
            ys_right.extend([snap.y_right for snap in snaps])
            
            for coll in ax.collections:
                coll.remove()
            
            # ax.collections.clear()
            ax.scatter(xs_cl, ys_cl, s=2, c='g', label="centerline")
            ax.scatter(xs_left, ys_left, s=1, c='r', label="left edge")
            ax.scatter(xs_right, ys_right, s=1, c='b', label="right edge")
            # ax.legend(loc="best")

            
            # last_segment_size = len(snaps)
            
            # if not append_last:
            #     xs_cl = xs_cl[:-last_segment_size] + [snap.x_cl for snap in snaps] + xs_cl[-last_segment_size:]
            #     ys_cl = ys_cl[:-last_segment_size] + [snap.y_cl for snap in snaps] + ys_cl[-last_segment_size:]
            #     xs_left = xs_left[:-last_segment_size] + [snap.x_left for snap in snaps] + xs_left[-last_segment_size:]
            #     ys_left = ys_left[:-last_segment_size] + [snap.y_left for snap in snaps] + ys_left[-last_segment_size:]
            #     xs_right = xs_right[:-last_segment_size] + [snap.x_right for snap in snaps] + xs_right[-last_segment_size:]
            #     ys_right = ys_right[:-last_segment_size] + [snap.y_right for snap in snaps] + ys_right[-last_segment_size:]
            # else:
            #     xs_cl.extend([snap.x_cl for snap in snaps])
            #     ys_cl.extend([snap.y_cl for snap in snaps])
            #     xs_left.extend([snap.x_left for snap in snaps])
            #     ys_left.extend([snap.y_left for snap in snaps])
            #     xs_right.extend([snap.x_right for snap in snaps])
            #     ys_right.extend([snap.y_right for snap in snaps])  
            
            for snap in snaps:
                print(snap)
                
            #     xs_cl.append(snap.x_cl); ys_cl.append(snap.y_cl)
            #     xs_left.append(snap.x_left); ys_left.append(snap.y_left)
            #     xs_right.append(snap.x_right); ys_right.append(snap.y_right)

            # # # draw
            # # line_cl.set_data(xs_cl, ys_cl)
            # # line_left.set_data(xs_left, ys_left)
            # # line_right.set_data(xs_right, ys_right)
            # _xs_cl = [snap.x_cl for snap in snaps]
            # _ys_cl = [snap.y_cl for snap in snaps]
            # _xs_left = [snap.x_left for snap in snaps]
            # _ys_left = [snap.y_left for snap in snaps]
            # _xs_right = [snap.x_right for snap in snaps]
            # _ys_right = [snap.y_right for snap in snaps]
            # ax.scatter(_xs_cl, _ys_cl, s=2, c='g', label="centerline")
            # ax.scatter(_xs_left, _ys_left, s=1, c='r', label="left edge")
            # ax.scatter(_xs_right, _ys_right, s=1, c='b', label="right edge")

            # if (last_plot_anchor is None or
            #     abs(snaps[0].x_cl-last_plot_anchor[0])>5 or
            #     abs(snaps[0].y_cl-last_plot_anchor[1])>5):
            #     ax.relim(); ax.autoscale_view()
            #     last_plot_anchor = (snaps[0].x_cl, snaps[0].y_cl)

            # schedule a redraw and process GUI events
            fig.canvas.draw_idle()
            fig.canvas.flush_events()

            # tiny sleep so the notebook kernel stays responsive
            time.sleep(0.05)

    except KeyboardInterrupt:
        print("\nStopping (KeyboardInterrupt)...")
    finally:
        stop_event.set()
        poller.join(timeout=2.0)
        # if logging, close file:
        # f.flush(); f.close()
        try:
            plt.close(fig)
        except Exception:
            pass


In [ ]:
run_logger_and_plot(rs,
    road_poll_sec=polling_rate,
    grid_N=11               # discretization density
)


# run_logger_and_plot(sensor,
#     road_poll_sec=0.05,
#     csv_path="track_log.csv",
#     use_poly_newton=True,   # set False to skip Newton path entirely
#     grid_N=41               # discretization density
# )

In [ ]:
run_logger_and_plot(rs,
    road_poll_sec=polling_rate,
    grid_N=11               # discretization density
)


# run_logger_and_plot(sensor,
#     road_poll_sec=0.05,
#     csv_path="track_log.csv",
#     use_poly_newton=True,   # set False to skip Newton path entirely
#     grid_N=41               # discretization density
# )

In [ ]:
# # ====================================
# # ---- Polling + CSV + Live Plot  ----
# # ====================================

# class SensorPoller(threading.Thread):
#     def __init__(self, sensor, poll_period_sec, out_queue, stop_event):
#         super().__init__(daemon=True)
#         self.sensor = sensor
#         self.dt = float(poll_period_sec)
#         self.q = out_queue
#         self.stop_event = stop_event

#     def run(self):
#         next_t = time.perf_counter()
#         while not self.stop_event.is_set():
#             try:
#                 data = self.sensor.poll()
#                 if data is not None:
#                     self.q.put(data)
#             except Exception as e:
#                 print(f"[poller] error: {e!r}")
#             next_t += self.dt
#             sleep = next_t - time.perf_counter()
#             if sleep > 0:
#                 time.sleep(sleep)
#             else:
#                 next_t = time.perf_counter()

# def run_logger_and_plot(
#     sensor, road_poll_sec=0.05, csv_path="track_log.csv",
#     use_poly_newton=True, grid_N=41
# ):
#     """
#     Segment-aware polling/logging/plotting.
#     - Resets p* when a new segment (xStartCL,yStartCL) appears.
#     - Computes discrete (no-Newton) and optional poly-accurate metrics.
#     """
#     q = Queue(maxsize=2048)
#     stop_event = threading.Event()
#     poller = SensorPoller(sensor, road_poll_sec, q, stop_event)
#     poller.start()

#     # # CSV
#     # fieldnames = [
#     #     "time","seg_id",
#     #     "x_cl","y_cl","z_cl",
#     #     "heading_cl_discrete","heading_cl_poly",
#     #     "kappa_discrete","kappa_poly","radius_discrete","radius_poly",
#     #     "w_left","w_right","halfWidth",
#     #     "dist2CL","dist2Left","dist2Right",
#     #     "speedLimit","drivability","flag1way","numlane",
#     #     "s_along","p_star_discrete","p_star_poly"
#     # ]
#     # f = open(csv_path, "w", newline="")
#     # writer = csv.DictWriter(f, fieldnames=fieldnames)
#     # writer.writeheader()

#     # Plot
#     plt.ion()
#     fig, ax = plt.subplots(figsize=(7, 7))
#     line_cl,   = ax.plot([], [], lw=2, label="centerline")
#     line_left, = ax.plot([], [], lw=1, alpha=0.7, label="left edge")
#     line_right,= ax.plot([], [], lw=1, alpha=0.7, label="right edge")
#     ax.set_aspect("equal")
#     ax.set_title("Live Centerline (segment-aware)")
#     ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")

#     xs_cl, ys_cl = [], []
#     xs_left, ys_left = [], []
#     xs_right, ys_right = [], []

#     last_s = 0.0
#     last_plot_anchor = None
#     tracker = SegmentTracker()

#     try:
#         while True:
#             try:
#                 rd = q.get(timeout=1.0)
#             except Empty:
#                 plt.pause(0.001); continue

#             snap = compute_snapshot(
#                 rd, last_s, tracker,
#                 use_poly_newton=use_poly_newton, grid_N=grid_N
#             )
#             last_s = snap.s_along

#             # # CSV
#             # writer.writerow({
#             #     "time": snap.t, "seg_id": snap.seg_id,
#             #     "x_cl": snap.x_cl, "y_cl": snap.y_cl, "z_cl": snap.z_cl,
#             #     "heading_cl_discrete": snap.heading_cl_discrete,
#             #     "heading_cl_poly": snap.heading_cl_poly,
#             #     "kappa_discrete": snap.kappa_discrete,
#             #     "kappa_poly": snap.kappa_poly,
#             #     "radius_discrete": snap.radius_discrete,
#             #     "radius_poly": snap.radius_poly,
#             #     "w_left": snap.w_left, "w_right": snap.w_right, "halfWidth": snap.halfWidth,
#             #     "dist2CL": snap.dist2CL, "dist2Left": snap.dist2Left, "dist2Right": snap.dist2Right,
#             #     "speedLimit": snap.speedLimit, "drivability": snap.drivability,
#             #     "flag1way": snap.flag1way, "numlane": snap.numlane,
#             #     "s_along": snap.s_along,
#             #     "p_star_discrete": snap.p_star_discrete, "p_star_poly": snap.p_star_poly
#             # })
#             # if (len(xs_cl) % 50) == 0:
#             #     f.flush()

#             # Plot traces
#             xs_cl.append(snap.x_cl); ys_cl.append(snap.y_cl)
#             # tangent (for normal) from last two CL points
#             if len(xs_cl) >= 2:
#                 dx = xs_cl[-1]-xs_cl[-2]; dy = ys_cl[-1]-ys_cl[-2]
#                 norm = math.hypot(dx, dy) or 1.0
#                 nx, ny = -dy/norm, dx/norm
#             else:
#                 nx, ny = -math.sin(snap.heading_cl_discrete), math.cos(snap.heading_cl_discrete)

#             xL = snap.x_cl + nx * snap.w_left
#             yL = snap.y_cl + ny * snap.w_left
#             xR = snap.x_cl - nx * snap.w_right
#             yR = snap.y_cl - ny * snap.w_right
#             xs_left.append(xL); ys_left.append(yL)
#             xs_right.append(xR); ys_right.append(yR)

#             line_cl.set_data(xs_cl, ys_cl)
#             line_left.set_data(xs_left, ys_left)
#             line_right.set_data(xs_right, ys_right)

#             if last_plot_anchor is None or (abs(snap.x_cl-last_plot_anchor[0])>5 or abs(snap.y_cl-last_plot_anchor[1])>5):
#                 ax.relim(); ax.autoscale_view()
#                 last_plot_anchor = (snap.x_cl, snap.y_cl)

#             plt.pause(0.001)

#     except KeyboardInterrupt:
#         print("\nStopping...")
#     finally:
#         stop_event.set()
#         poller.join(timeout=2.0)
#         # f.flush(); f.close()
#         plt.ioff(); plt.show()

In [ ]:
for _i in range(5):
    draw = rs.poll()
    print(draw)

In [ ]:
import matplotlib
%matplotlib qt
# matplotlib.use("qtAgg")  # Or another interactive backend like "TkAgg", Qt5Agg
# matplotlib.use("widget")  # Or another interactive backend like "TkAgg", Qt5Agg
import matplotlib.pyplot as plt

In [ ]:
from time import sleep
from beamngpy import BeamNGpy, Scenario, Vehicle, set_up_simple_logging, angle_to_quat
from beamngpy.misc.vec3 import vec3
from matplotlib.animation import FuncAnimation

In [ ]:
GRANULARITY = 100


# Evaluates the road parametric cubic polynomials.
def discretize(d):
    eval_range = GRANULARITY + 1
    granularity_inv = 1.0 / GRANULARITY
    # Cache the individual coefficients of the u(p), v(p) equations, for the road centerline, the road left edge, and the road right edge.
    aU_CL, bU_CL, cU_CL, dU_CL = d["uAofCL"], d["uBofCL"], d["uCofCL"], d["uDofCL"]
    aV_CL, bV_CL, cV_CL, dV_CL = d["vAofCL"], d["vBofCL"], d["vCofCL"], d["vDofCL"]
    aU_LE, bU_LE, cU_LE, dU_LE = (
        d["uAofLeftRE"],
        d["uBofLeftRE"],
        d["uCofLeftRE"],
        d["uDofLeftRE"],
    )
    aV_LE, bV_LE, cV_LE, dV_LE = (
        d["vAofLeftRE"],
        d["vBofLeftRE"],
        d["vCofLeftRE"],
        d["vDofLeftRE"],
    )
    aU_RE, bU_RE, cU_RE, dU_RE = (
        d["uAofRightRE"],
        d["uBofRightRE"],
        d["uCofRightRE"],
        d["uDofRightRE"],
    )
    aV_RE, bV_RE, cV_RE, dV_RE = (
        d["vAofRightRE"],
        d["vBofRightRE"],
        d["vCofRightRE"],
        d["vDofRightRE"],
    )

    # Will evaluate each parametric cubic, discretized at the chosen granularity.
    pts_CL, pts_LE, pts_RE = [], [], []
    for i in range(eval_range):
        p = i * granularity_inv  # The parameter p, in [0, 1], and its powers.
        p2 = p * p
        p3 = p2 * p

        # Evaluate the polynomials.
        u_CL = (
            aU_CL + (p * bU_CL) + (p2 * cU_CL) + (p3 * dU_CL)
        )  # Road centerline u(p), v(p).
        v_CL = aV_CL + (p * bV_CL) + (p2 * cV_CL) + (p3 * dV_CL)
        u_LE = (
            aU_LE + (p * bU_LE) + (p2 * cU_LE) + (p3 * dU_LE)
        )  # Road left edge u(p), v(p).
        v_LE = aV_LE + (p * bV_LE) + (p2 * cV_LE) + (p3 * dV_LE)
        u_RE = (
            aU_RE + (p * bU_RE) + (p2 * cU_RE) + (p3 * dU_RE)
        )  # Road right edge u(p), v(p).
        v_RE = aV_RE + (p * bV_RE) + (p2 * cV_RE) + (p3 * dV_RE)

        pts_CL.append(
            vec3(u_CL, v_CL) + vec3(d["xStartCL"], d["yStartCL"])
        )  # Road centerline.
        pts_LE.append(
            vec3(u_LE, v_LE) + vec3(d["xStartL"], d["yStartL"])
        )  # Road left edge (with offset added horizontally).
        pts_RE.append(
            vec3(u_RE, v_RE) + vec3(d["xStartR"], d["yStartR"])
        )  # Road right edge (with offset added horizontally).

    return pts_CL, pts_LE, pts_RE

In [ ]:
def update():
    sleep(1)
    print("in while...")
    d_raw = rs.poll()
    if not d_raw:
        print("No data received.")
        return
    d = d_raw[0]
    print("got raw data...")
    pts_CL, pts_LE, pts_RE = discretize(d)
    print("got discretized data...")
    ax.set(xlabel="x", ylabel="y", title="Road Profile")
    ax.set_aspect("equal", adjustable="box")
    print("to plot...")
    x, y = [], []  # Plot the road centerline.
    for i in range(len(pts_CL)):
        p = pts_CL[i]
        x.append(p.x)
        y.append(p.y)
    ax.plot(x, y, "b")
    x, y = [], []  # Plot the road left edge.
    for i in range(len(pts_LE)):
        p = pts_LE[i]
        x.append(p.x)
        y.append(p.y)
    ax.plot(x, y, "r")
    x, y = [], []  # Plot the road right edge.
    for i in range(len(pts_RE)):
        p = pts_RE[i]
        x.append(p.x)
        y.append(p.y)
    ax.plot(x, y, "g")
    plt.legend(["Centerline", "Left Edge", "Right Edge"])
    print("to get vehicle state...")
    vehicle.sensors.poll()  # Plot vehicle position and direction.
    pos = vehicle.state["pos"]
    dir = vehicle.state["dir"]
    vp = vec3(pos[0], pos[1])
    v1 = vp + vec3.normalize(vec3(dir[0], dir[1])) * 10.0
    v2 = v1 - vp
    ax.plot(vp.x, vp.y, "bo")
    plt.arrow(vp.x, vp.y, v2.x, v2.y, width=0.05)

    frames.append([ax.plot])

In [ ]:
vehicle = veh
vehicle.ai.set_mode("disabled")
sleep(1.0)
fig, ax = plt.subplots()
frames = []

In [ ]:
# Create the animation frames
for k in range(200):
    update()

# Create the animation
animation = FuncAnimation(fig, update, frames, interval=50)
